In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

In [2]:
df = pd.read_csv(
    "../data/processed/creditwise_cleaned.csv"
)

final_model = joblib.load(
    "../data/outputs/final_model.pkl"
)

X_test_processed = joblib.load(
    "../data/outputs/X_test_processed.pkl"
)

y_test = joblib.load(
    "../data/outputs/y_test.pkl"
)

print("Dataset:", df.shape)
print("Test data:", X_test_processed.shape)

Dataset: (12000, 44)
Test data: (2400, 77)


In [3]:
predictions = final_model.predict(
    X_test_processed
)

probabilities = final_model.predict_proba(
    X_test_processed
)[:, 1]

print("Predictions generated.")

Predictions generated.


In [4]:
monitoring_metrics = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "ROC_AUC",
        "PR_AUC"
    ],
    "Value": [
        accuracy_score(y_test, predictions),
        precision_score(
            y_test,
            predictions,
            zero_division=0
        ),
        recall_score(
            y_test,
            predictions,
            zero_division=0
        ),
        f1_score(
            y_test,
            predictions,
            zero_division=0
        ),
        roc_auc_score(
            y_test,
            probabilities
        ),
        average_precision_score(
            y_test,
            probabilities
        )
    ]
})

monitoring_metrics

,Metric,Value
0,Accuracy,0.753750
1,Precision,0.665445
2,Recall,0.762605
3,F1,0.710720
4,ROC_AUC,0.848509
5,PR_AUC,0.792109


In [5]:
prediction_distribution = pd.DataFrame({
    "Prediction": [0, 1],
    "Count": [
        (predictions == 0).sum(),
        (predictions == 1).sum()
    ]
})

prediction_distribution["Percentage"] = (
    prediction_distribution["Count"]
    / len(predictions)
    * 100
)

prediction_distribution

,Prediction,Count,Percentage
0,0,1309,54.541667
1,1,1091,45.458333


In [6]:
actual_rate = y_test.mean() * 100
predicted_rate = predictions.mean() * 100

distribution_comparison = pd.DataFrame({
    "Measure": [
        "Actual historical approval rate",
        "Predicted approval rate"
    ],
    "Percentage": [
        actual_rate,
        predicted_rate
    ]
})

distribution_comparison

,Measure,Percentage
0,Actual historical approval rate,39.666667
1,Predicted approval rate,45.458333


In [7]:
probability_summary = pd.Series(
    probabilities
).describe()

probability_summary

count    2400.000000
mean        0.464760
std         0.297648
min         0.000257
25%         0.199426
50%         0.450544
75%         0.718752
max         0.999417
dtype: float64

In [8]:
feature_monitoring = pd.DataFrame({
    "Feature": [
        "credit_score",
        "debt_to_income_ratio",
        "credit_utilization_ratio",
        "annual_income",
        "years_employed",
        "late_payments_24m"
    ],
    "Mean": [
        df["credit_score"].mean(),
        df["debt_to_income_ratio"].mean(),
        df["credit_utilization_ratio"].mean(),
        df["annual_income"].mean(),
        df["years_employed"].mean(),
        df["late_payments_24m"].mean()
    ],
    "Median": [
        df["credit_score"].median(),
        df["debt_to_income_ratio"].median(),
        df["credit_utilization_ratio"].median(),
        df["annual_income"].median(),
        df["years_employed"].median(),
        df["late_payments_24m"].median()
    ],
    "Missing": [
        df["credit_score"].isna().sum(),
        df["debt_to_income_ratio"].isna().sum(),
        df["credit_utilization_ratio"].isna().sum(),
        df["annual_income"].isna().sum(),
        df["years_employed"].isna().sum(),
        df["late_payments_24m"].isna().sum()
    ]
})

feature_monitoring

,Feature,Mean,Median,Missing
0,credit_score,654.067167,654.0000,0
1,debt_to_income_ratio,0.228380,0.2270,0
2,credit_utilization_ratio,0.222521,0.2192,0
3,annual_income,54987.883333,49400.0000,0
4,years_employed,5.208025,3.4000,0
5,late_payments_24m,0.048667,0.0000,0


In [9]:
monitoring_metrics["Status"] = np.where(
    monitoring_metrics["Value"].notna(),
    "Available",
    "Check"
)

monitoring_metrics

,Metric,Value,Status
0,Accuracy,0.753750,Available
1,Precision,0.665445,Available
2,Recall,0.762605,Available
3,F1,0.710720,Available
4,ROC_AUC,0.848509,Available
5,PR_AUC,0.792109,Available


In [10]:
monitoring_metrics.to_csv(
    "../data/outputs/model_monitoring_metrics.csv",
    index=False
)

prediction_distribution.to_csv(
    "../data/outputs/prediction_distribution.csv",
    index=False
)

distribution_comparison.to_csv(
    "../data/outputs/prediction_distribution_comparison.csv",
    index=False
)

feature_monitoring.to_csv(
    "../data/outputs/feature_monitoring.csv",
    index=False
)

print("Monitoring outputs saved successfully.")

Monitoring outputs saved successfully.
